In [27]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

# 멀티모달 검색기(Multimodal Retriever)구현

In [8]:
from typing import List
from langchain_core.documents import Document
import voyageai

from qdrant_client import QdrantClient
from qdrant_client.http import models
import uuid


qdrant_client = QdrantClient("localhost", port=6333, grpc_port=True)
collection_name = "Practice_Multimodal_retriever"
voyageai_client = voyageai.Client()


class MultimodalRetriever:
    def __init__(self, qdrant_client, collection_name:str, votageai_client, limit: int = 5):
        self.qdrant_client = qdrant_client
        self.collection_name = collection_name
        self.voyageai_client = votageai_client
        self.limit = limit # 검색해서 뽑아낼 문서의 개수

    def invoke(self, query:str) -> List[Document]:
        """질의에 대해 관련 문서와 이미지를 검색하여 Document 형태로 변환"""
        query_result = self.voyageai_client.multimodal_embed(
            inputs=[query],
            model="voyage-multimodal-3",
            input_type="query" # 사용자의 질문이기 때문에
        )        
        query_vector = query_result.embeddings[0] #query vector를 임베딩의 첫번쨰 값으로 저장

        # Qdrant에서 검색
        search_result = self.qdrant_client.search(
            collection_name=self.collection_name,
            query_vector=query_vector,
            limit=self.limit
        )

        documents = []
        
        # langchain의 Document 형식으로 변환
        for hit in search_result:
            doc = Document(
                page_content=hit.payload.get("text", ""), # 이미지 내용
                metadata={
                    'score': hit.score, # 유사도 점수
                    'image_path': hit.payload.get('image_path', ''),
                    **hit.payload
                }
            )
            documents.append(doc)
        return documents       

In [9]:
retriever = MultimodalRetriever(
    qdrant_client=qdrant_client,
    collection_name=collection_name,
    votageai_client=voyageai_client,
    limit=5
)

query = ["회사 로고 많은 페이지"]

retriever.invoke(query)

C:\Users\skyop\AppData\Local\Temp\ipykernel_3392\1206930017.py:32: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_result = self.qdrant_client.search(


[Document(metadata={'score': 0.23337837, 'image_path': './Scratch2\\physical_ai_report-47.png', 'source': 'voyage', 'text': ''}, page_content=''),
 Document(metadata={'score': 0.228598, 'image_path': './Scratch2\\physical_ai_report-48.png', 'source': 'voyage', 'text': ''}, page_content=''),
 Document(metadata={'score': 0.20137338, 'image_path': './Scratch2\\physical_ai_report-16.png', 'source': 'voyage', 'text': ''}, page_content=''),
 Document(metadata={'score': 0.19793469, 'image_path': './Scratch2\\physical_ai_report-26.png', 'source': 'voyage', 'text': "<!-- image -->\n\n* 출처 : https://www.figure.ai/\n\n- 2025년 2월, 자사 로봇의 인식, 언어 이해, 제어 기능을 통합하여 기존 로봇 공학의 한계를 극복하기 위해 시각-언어-행동(VLA, Vision-Language-Action) 모델 '헬릭스(Helix)'를 공개(Figure.ai, 2025. 2.)\n- 기존에는 로봇에게 새로운 행동을 가르치려면 전문가의 수동 프로그래밍이나 수천 번의 시뮬레이션이 필요했으나, 헬릭스는 로봇이 카메라로 수집한 시각 정보와 자연어 명령을 결합하여 이전에 접한 적 없는 물체도 실시간으로 조작\n- 작업별 미세조정 없이 단일 신경망으로 모든 동작을 학습하는 한편, 기기 내장형 저전력 GPU에서 실행 할 수 있어 신속한 상용화를 지원\n- 시스템 2(S2)와 시스템 1(S1)이라는 2개의 시스템을 통

# RAG 시스템 구현

- 이미지 값을 base64로 인코딩 하여야 llm에게 넘겨 줄수 있다는것을 명심!
- 멀티모달을 지원하는 LLM을 호출해야 하는것을 명심!

In [10]:
import os
import base64

def encode_inage_to_base64(image_path):
    """이미지를 base64로 인코딩"""
    if os.path.exists(image_path):
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser